# The Python UI Revolution: Rapid Prototyping with Gradio

An AI Architect must deliver value quickly. **Gradio** allows us to bypass the "Frontend Bottleneck." We can turn a Python function into a shareable web app in under 10 lines of code. No HTML, no CSS, no JavaScript—just pure logic.

In [2]:
import gradio as gr
from litellm import completion
from dotenv import load_dotenv
import os

load_dotenv(override=True)

print("🎨 Gradio Environment Initialized!")

🎨 Gradio Environment Initialized!


## 1. The "Hello World" of AI Interfaces
We start by wrapping a simple LiteLLM query into a Gradio `Interface`. This creates a text-in, text-out web app instantly.

In [ ]:
def analyze_text(user_input):
    """Our Backend Logic"""
    response = completion(
        model="openai/gpt-4o-mini",
        messages=[{"role": "user", "content": f"Summarize this in one sentence: {user_input}"}]
    )
    return response.choices[0].message.content

# The UI Layer
demo = gr.Interface(
    fn=analyze_text, 
    inputs="text", 
    outputs="text",
    title="AI Architect: Instant Summarizer"
)

# To view this, we would run: demo.launch()
demo.launch()

## 2. The Professional Chatbot: `gr.ChatInterface`
For most AI applications, we need a conversational UI. Gradio provides a specialized `ChatInterface` that handles the chat history (memory) for us automatically.

In [ ]:
def predict(message, history):
    """
    Gradio ChatInterface automatically passes 'history' as a list of [user, assistant] pairs.
    """
    # Convert Gradio history to OpenAI message format
    messages = [{"role": "system", "content": "You are a helpful AI Architect."}]

    for msg in history:
        if hasattr(msg, 'role'): # It's a Message object
            messages.append({"role": msg.role, "content": msg.content})
        elif isinstance(msg, dict): # It's a dictionary
            messages.append({"role": msg['role'], "content": msg['content']})
        else: # Handle older list of [user, bot] pairs if fallback is needed
            user_msg, assistant_msg = msg
            messages.append({"role": "user", "content": user_msg})
            messages.append({"role": "assistant", "content": assistant_msg})
    
    messages.append({"role": "user", "content": message})

    response = completion(model="openai/gpt-4o-mini", messages=messages)
    return response.choices[0].message.content

chat_demo = gr.ChatInterface(
    fn=predict, 
    title="Architect Chat v1",
    description="Ask me anything about LLM Engineering!"
)

chat_demo.launch()

## 3. High-Fidelity Customization: `gr.Blocks`
When you need a custom layout (columns, rows, buttons), we use `gr.Blocks`. This is the "Pro Mode" of Gradio.

In [7]:
with gr.Blocks() as pro_demo:
    gr.Markdown("# AI Model Comparison Lab")
    
    with gr.Row():
        with gr.Column():
            input_text = gr.Textbox(label="Enter Prompt")
            submit_btn = gr.Button("Battle!", variant="primary")
        
        with gr.Column():
            output_openai = gr.Textbox(label="OpenAI Result")
            output_deepseek = gr.Textbox(label="DeepSeek Result")

    def run_battle(text):
        res1 = completion(model="openai/gpt-5", messages=[{"role": "user", "content": text}])
        res2 = completion(model="deepseek/deepseek-chat", messages=[{"role": "user", "content": text}])
        return res1.choices[0].message.content, res2.choices[0].message.content

    submit_btn.click(fn=run_battle, inputs=input_text, outputs=[output_openai, output_deepseek])

print("🚀 Professional 'Blocks' UI is defined.")
pro_demo.launch()

🚀 Professional 'Blocks' UI is defined.
* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.
